# Семинар 3. Организация обучения в PyTorch

Основной трек: Chapter 2 книги Daniel Voigt Godoy. Продолжаем линейную регрессию
второго семинара: выделяем шаг обучения, вводим батчи, валидацию, TensorBoard и
контрольные точки. Затем отдельно разбираем приёмы, необходимые для ДЗ1.

## Содержание

1. [Знакомый цикл обучения](#section-1)
2. [Функция шага обучения](#section-2)
3. [Dataset и TensorDataset](#section-3)
4. [DataLoader](#section-4)
5. [Мини-батчи и эпохи](#section-5)
6. [Разбиение и валидация](#section-6)
7. [TensorBoard](#section-7)
8. [Сохранение, возобновление и предсказания](#section-8)
9. [Собираем всё вместе](#section-9)
10. [К ДЗ1: данные и формы](#section-10)
11. [К ДЗ1: собственный backward](#section-11)
12. [К ДЗ1: классификация и эксперименты](#section-12)
13. [Задания](#section-13)

In [ ]:
from pathlib import Path
from runpy import run_path

# При согласованной проверке Colab замените main на SHA коммита.
course_revision = 'main'
try:
    import google.colab
except ModuleNotFoundError:
    repository_root = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / 'config.py').is_file() and (p / 'plots' / 'seminar03.py').is_file()), None)
    if repository_root is None:
        raise RuntimeError('Откройте notebook из локальной копии 2026_ML3.')
else:
    repository_root = Path.cwd().resolve()
    from urllib.request import urlopen
    url = f'https://raw.githubusercontent.com/SergeyMalashenko/2026_ML3/{course_revision}/config.py'
    with urlopen(url, timeout=30) as response:
        config_source = response.read()
    compile(config_source, 'config.py', 'exec')
    (repository_root / 'config.py').write_bytes(config_source)
course_config = run_path(str(repository_root / 'config.py'))
seminar_plots = course_config['config_seminar03'](branch=course_revision)
select_device = course_config['select_device']

from tempfile import TemporaryDirectory
if 'lesson_workspace' in globals():
    lesson_workspace.cleanup()
lesson_workspace = TemporaryDirectory(prefix='ml3-seminar03-')
lesson_dir = Path(lesson_workspace.name)
for folder in ('data_preparation', 'model_configuration', 'model_training'):
    (lesson_dir / folder).mkdir()

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
from tempfile import mkdtemp

seminar_plots.configure_plots()
plot_losses = seminar_plots.plot_losses
plot_resumed_losses = seminar_plots.plot_resumed_losses
device = select_device()  # Для воспроизводимой проверки: select_device('cpu').
print('Устройство:', device)

<a id="section-1"></a>

## 1. Знакомый цикл обучения

Модель, функция потерь и оптимизатор уже знакомы. Сначала выполним прежний цикл целиком. Затем будем менять его организацию, сохраняя четыре шага.

### Обучение V0

In [ ]:
true_b = 1
true_w = 2
N = 100
np.random.seed(42)
x = np.random.rand(N, 1)
epsilon = .1 * np.random.randn(N, 1)
y = true_b + true_w * x + epsilon

idx = np.arange(N)
np.random.shuffle(idx)
train_idx = idx[:int(N * .8)]
val_idx = idx[int(N * .8):]
x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]
seminar_plots.plot_data(x_train, y_train, x_val, y_val);

In [ ]:
%%writefile "{lesson_dir}/data_preparation/v0.py"
device = select_device()
x_train_tensor = torch.as_tensor(x_train).float().to(device)
y_train_tensor = torch.as_tensor(y_train).float().to(device)

In [ ]:
%%writefile "{lesson_dir}/model_configuration/v0.py"
device = select_device()
lr = 0.1
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(1, 1)).to(device)
optimizer = optim.SGD(model.parameters(), lr=lr)
loss_fn = nn.MSELoss(reduction='mean')

In [ ]:
%run -i "{lesson_dir}/data_preparation/v0.py"
%run -i "{lesson_dir}/model_configuration/v0.py"

In [ ]:
# %load model_training/v0.py

# Количество эпох
n_epochs = 1000

for epoch in range(n_epochs):
    # Режим обучения
    model.train()

    # Шаг 1: предсказания, прямой проход
    # Предсказания вычисляет модель
    yhat = model(x_train_tensor)
    
    # Шаг 2: функция потерь
    loss = loss_fn(yhat, y_train_tensor)

    # Шаг 3: градиенты параметров модели
    loss.backward()
    
    # Шаг 4: обновление параметров
    optimizer.step() # SGD -> another optimizator
    optimizer.zero_grad()

### Справка: шаг L-BFGS и closure

<img src="https://raw.githubusercontent.com/SergeyMalashenko/2026_ML3/main/notebooks/03_neuralnetwork/images/L-BFGS.png" alt="Справка: шаг L-BFGS и closure" width="1000">

[Открыть схему в полном размере](https://raw.githubusercontent.com/SergeyMalashenko/2026_ML3/main/notebooks/03_neuralnetwork/images/L-BFGS.png)

Это справка, а не замена SGD в основном примере. Один `step(closure)` может включать несколько внутренних итераций и повторных вычислений. Линейный поиск включается отдельно: `line_search_fn="strong_wolfe"`; по умолчанию `line_search_fn=None`.

In [ ]:
print(model.state_dict())

<a id="section-2"></a>

## 2. Функция шага обучения

Внутри каждой эпохи повторяются одни и те же действия. Для заданных модели,
потерь и оптимизатора меняются только данные. Сначала разберём более простой
пример: функцию, которая строит другую функцию.

In [ ]:
def square(x):
    return x ** 2

def cube(x):
    return x ** 3

def fourth_power(x):
    return x ** 4

# И так далее

In [ ]:
def generic_exponentiation(x, exponent):
    return x ** exponent

In [ ]:
def skeleton_exponentiation(x):
    return x ** exponent

In [ ]:
try:
    skeleton_exponentiation(2)
except NameError as error:
    print('Ожидаемая ошибка:', error)

In [ ]:
def exponentiation_builder(exponent):
    def skeleton_exponentiation(x):
        return x ** exponent

    return skeleton_exponentiation

In [ ]:
returned_function = exponentiation_builder(2)

returned_function

In [ ]:
returned_function(5)

In [ ]:
square = exponentiation_builder(2)
cube = exponentiation_builder(3)
fourth_power = exponentiation_builder(4)

# И так далее

### Шаг обучения

Что фиксируем при создании функции? Что передаём при каждом вызове? Возвращается число потерь **до обновления**, а параметры уже изменены.

In [ ]:
def make_train_step_fn(model, loss_fn, optimizer):
    # Создаём функцию шага обучения
    def perform_train_step_fn(x, y):
        # Режим обучения
        model.train()
        
        # Шаг 1: предсказания, прямой проход
        yhat = model(x)
        # Шаг 2: функция потерь
        loss = loss_fn(yhat, y)
        # Шаг 3: градиенты параметров модели
        loss.backward()
        # Шаг 4: обновление параметров
        optimizer.step()
        optimizer.zero_grad()
        
        # Возвращаем число потерь
        return loss.item()
    
    # Возвращаем функцию шага обучения
    return perform_train_step_fn

### Конфигурация модели V1

In [ ]:
%run -i "{lesson_dir}/data_preparation/v0.py"

In [ ]:
%%writefile "{lesson_dir}/model_configuration/v1.py"

device = select_device()

# Скорость обучения
lr = 0.1

torch.manual_seed(42)
# Создаём модель и переносим её на выбранное устройство
model = nn.Sequential(nn.Linear(1, 1)).to(device)

# Оптимизатор получает параметры этой модели
optimizer = optim.SGD(model.parameters(), lr=lr)

# Средний квадрат ошибки
loss_fn = nn.MSELoss(reduction='mean')

# Создаём шаг для этой модели, функции потерь и оптимизатора
train_step_fn = make_train_step_fn(model, loss_fn, optimizer)

In [ ]:
%run -i "{lesson_dir}/model_configuration/v1.py"

In [ ]:
train_step_fn

### Обучение V1

In [ ]:
%%writefile "{lesson_dir}/model_training/v1.py"

# Количество эпох
n_epochs = 1000

losses = []

# Для каждой эпохи
for epoch in range(n_epochs):
    # Выполняем шаг и получаем потери
    loss = train_step_fn(x_train_tensor, y_train_tensor)
    losses.append(loss)

In [ ]:
%run -i "{lesson_dir}/model_training/v1.py"

In [ ]:
# Проверяем параметры модели
print(model.state_dict())

<a id="section-3"></a>

## 3. Dataset и TensorDataset

`Dataset` описывает получение одного примера. В map-style варианте реализуем `__getitem__` и `__len__`. Данные пока остаются на CPU; на ускоритель будем переносить очередной батч.

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, x_tensor, y_tensor):
        self.x = x_tensor
        self.y = y_tensor
        
    def __getitem__(self, index):
        return (self.x[index], self.y[index])

    def __len__(self):
        return len(self.x)

# Почему здесь CPU? Переносить будем очередной батч
x_train_tensor = torch.from_numpy(x_train).float()
y_train_tensor = torch.from_numpy(y_train).float()

train_data = CustomDataset(x_train_tensor, y_train_tensor)
print(train_data[0])

### TensorDataset

Если признаки и ответы уже представлены согласованными тензорами, собственный класс не нужен. `TensorDataset` индексирует их вместе.

In [ ]:
train_data = TensorDataset(x_train_tensor, y_train_tensor)
print(train_data[0])

<a id="section-4"></a>

## 4. DataLoader

`DataLoader` выбирает индексы, получает примеры из Dataset и объединяет их в батчи. `shuffle=True` меняет порядок обхода, но не соответствие признаков и ответов.

**Вопрос:** сколько батчей получится из 80 объектов при `batch_size=16`?

In [ ]:
train_loader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)

In [ ]:
next(iter(train_loader))

In [ ]:
print('Объектов:', len(train_data), '| Батчей:', len(train_loader))

### Подготовка данных V1

In [ ]:
%%writefile "{lesson_dir}/data_preparation/v1.py"

# Преобразуем массивы NumPy в тензоры
x_train_tensor = torch.from_numpy(x_train).float()
y_train_tensor = torch.from_numpy(y_train).float()

# Создаём Dataset
train_data = TensorDataset(x_train_tensor, y_train_tensor)

# Создаём DataLoader
train_loader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)

In [ ]:
%run -i "{lesson_dir}/data_preparation/v1.py"

<a id="section-5"></a>

## 5. Мини-батчи и эпохи

Одна эпоха теперь содержит несколько шагов оптимизатора. Внешний цикл перебирает эпохи, внутренний перебирает батчи. Данные и модель должны находиться на одном устройстве.

In [ ]:
%run -i "{lesson_dir}/model_configuration/v1.py"

In [ ]:
%%writefile "{lesson_dir}/model_training/v2.py"

# Количество эпох
n_epochs = 1000

losses = []

# Для каждой эпохи
for epoch in range(n_epochs):
    # Внутренний цикл по батчам
    mini_batch_losses = []
    batch_sizes = []
    for x_batch, y_batch in train_loader:
        # Dataset и полученные из него батчи находятся на CPU
        # Переносим очередной батч на устройство модели
        # 
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        # Выполняем шаг и получаем потери 
        # для этого батча
        mini_batch_loss = train_step_fn(x_batch, y_batch)
        mini_batch_losses.append(mini_batch_loss)
        batch_sizes.append(len(x_batch))

    # Среднее по объектам, учитываем размер батча
    # Потери по ходу эпохи
    loss = float(np.average(mini_batch_losses, weights=batch_sizes))
    
    losses.append(loss)

In [ ]:
%run -i "{lesson_dir}/model_training/v2.py"

In [ ]:
# Проверяем параметры модели
print(model.state_dict())

### Выделяем внутренний цикл

Для `reduction='mean'` и одного ответа на объект потери эпохи усредняем с
весами, равными размерам батчей. Простое среднее средних корректно только для
одинаковых размеров. Это уточнение к реализации Годоя.

**Вопрос:** у нас батчи из 16 и 4 объектов с MSE 1 и 9. Почему среднее 5 неверно?

Обучающие потери измеряются по ходу эпохи при меняющихся параметрах, а не
являются MSE одной финальной модели на всей обучающей выборке.

In [ ]:
def mini_batch(device, data_loader, step_fn):
    mini_batch_losses = []
    batch_sizes = []
    for x_batch, y_batch in data_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        mini_batch_loss = step_fn(x_batch, y_batch)
        mini_batch_losses.append(mini_batch_loss)
        batch_sizes.append(len(x_batch))

    loss = np.average(mini_batch_losses, weights=batch_sizes)
    return float(loss)

### Обучение V3

In [ ]:
%run -i "{lesson_dir}/data_preparation/v1.py"
%run -i "{lesson_dir}/model_configuration/v1.py"

In [ ]:
%%writefile "{lesson_dir}/model_training/v3.py"

# Количество эпох
n_epochs = 200

losses = []

for epoch in range(n_epochs):
    # Внутренний цикл по батчам
    loss = mini_batch(device, train_loader, train_step_fn)
    losses.append(loss)

In [ ]:
%run -i "{lesson_dir}/model_training/v3.py"

In [ ]:
# Проверяем параметры модели
print(model.state_dict())

<a id="section-6"></a>

## 6. Разбиение и валидация

`random_split` разделяет индексы Dataset, а не перемешивает батчи. Сейчас получим новое воспроизводимое разбиение с seed 13; оно отличается от NumPy-разбиения в начале.

### Подготовка данных V2

In [ ]:
%%writefile "{lesson_dir}/data_preparation/v2.py"

torch.manual_seed(13)

# Преобразуем в тензоры до разбиения
x_tensor = torch.from_numpy(x).float()
y_tensor = torch.from_numpy(y).float()

# Dataset для всей выборки
dataset = TensorDataset(x_tensor, y_tensor)

# Разделяем индексы
ratio = .8
n_total = len(dataset)
n_train = int(n_total * ratio)
n_val = n_total - n_train

train_data, val_data = random_split(dataset, [n_train, n_val])

# Загрузчик для каждой части
train_loader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(dataset=val_data, batch_size=16)

In [ ]:
%run -i "{lesson_dir}/data_preparation/v2.py"

In [ ]:
assert not set(train_data.indices) & set(val_data.indices)
assert len(train_data) + len(val_data) == len(dataset)
print('Размеры батчей валидации:', [len(xb) for xb, _ in val_loader])

### Оценка модели

`eval()` переключает поведение таких слоёв, как Dropout; сам по себе он
не запрещает запись графа. `no_grad()` запрещает запись графа в своём контексте,
но не переключает режим слоёв. При валидации используем оба.

**Вопрос:** какие два действия исчезают из шага валидации по сравнению с обучением?

### Два независимых переключателя: eval и no_grad

<img src="https://raw.githubusercontent.com/SergeyMalashenko/2026_ML3/main/notebooks/03_neuralnetwork/images/no_grad_and_model_eval.png" alt="Два независимых переключателя: eval и no_grad" width="1000">

[Открыть схему в полном размере](https://raw.githubusercontent.com/SergeyMalashenko/2026_ML3/main/notebooks/03_neuralnetwork/images/no_grad_and_model_eval.png)

Запись графа сама по себе ещё не вычисляет градиенты: для этого нужен `backward()` или `autograd.grad()`. Здесь речь об обычном обратном режиме autograd. Для BatchNorm схема предполагает стандартное `track_running_stats=True`.

In [ ]:
def make_val_step_fn(model, loss_fn):
    # Создаём функцию шага валидации
    def perform_val_step_fn(x, y):
        # Режим оценки
        model.eval()
        
        # Шаг 1: предсказания, прямой проход
        yhat = model(x)
        # Шаг 2: функция потерь
        loss = loss_fn(yhat, y)
        # Шагов 3 и 4 нет: при оценке параметры не обновляем
        return loss.item()
    
    return perform_val_step_fn

### Конфигурация модели V2

In [ ]:
%%writefile "{lesson_dir}/model_configuration/v2.py"

device = select_device()

# Скорость обучения
lr = 0.1

torch.manual_seed(42)
# Создаём модель и переносим её на выбранное устройство
model = nn.Sequential(nn.Linear(1, 1)).to(device)

# Оптимизатор получает параметры этой модели
optimizer = optim.SGD(model.parameters(), lr=lr)

# Средний квадрат ошибки
loss_fn = nn.MSELoss(reduction='mean')

# Создаём шаг для этой модели, функции потерь и оптимизатора
train_step_fn = make_train_step_fn(model, loss_fn, optimizer)

# Создаём шаг валидации
val_step_fn = make_val_step_fn(model, loss_fn)

In [ ]:
%run -i "{lesson_dir}/model_configuration/v2.py"

### Обучение V4

In [ ]:
%%writefile "{lesson_dir}/model_training/v4.py"

# Количество эпох
n_epochs = 200

losses = []
val_losses = []

for epoch in range(n_epochs):
    # Внутренний цикл по батчам
    loss = mini_batch(device, train_loader, train_step_fn)
    losses.append(loss)
    
    # Валидация
    # При валидации граф не записываем
    with torch.no_grad():
        val_loss = mini_batch(device, val_loader, val_step_fn)
        val_losses.append(val_loss)

In [ ]:
%run -i "{lesson_dir}/model_training/v4.py"

In [ ]:
# Проверяем параметры модели
print(model.state_dict())

### Кривые потерь

In [ ]:
fig = plot_losses(losses, val_losses)

<a id="section-7"></a>

## 7. TensorBoard

`SummaryWriter` записывает события, TensorBoard читает их и показывает графики.
Сначала отправим вычислительный граф и одну пару значений, затем перенесём запись
потерь внутрь цикла. `global_step` здесь означает индекс эпохи.

Учебные модули, события и checkpoint сохраняются во временном каталоге вне
репозитория. Они нужны для демонстрации, а не для долговременного хранения.
Для реального эксперимента выберите собственное постоянное место хранения.

In [ ]:
tensorboard_dir = lesson_dir / 'runs'
tensorboard_dir.mkdir(exist_ok=True)
show_tensorboard = True

In [ ]:
if show_tensorboard:
    %load_ext tensorboard
    %tensorboard --logdir "$tensorboard_dir"

### SummaryWriter

In [ ]:
writer = SummaryWriter(mkdtemp(prefix='test-', dir=tensorboard_dir))

### Граф модели

In [ ]:
# writer.add_graph(model)

In [ ]:
# Получаем признаки и ответы одного батча
sample_x, sample_y = next(iter(train_loader))

# Передаём пример входа на устройство модели
# Модель и данные должны находиться на одном устройстве
writer.add_graph(model, sample_x.to(device))

### Скалярные значения

In [ ]:
writer.add_scalars('loss', {'training': loss, 'validation': val_loss}, epoch)
writer.close()

### Конфигурация модели V3

In [ ]:
%run -i "{lesson_dir}/data_preparation/v2.py"

In [ ]:
%%writefile "{lesson_dir}/model_configuration/v3.py"

device = select_device()

# Скорость обучения
lr = 0.1

torch.manual_seed(42)
# Создаём модель и переносим её на выбранное устройство
model = nn.Sequential(nn.Linear(1, 1)).to(device)

# Оптимизатор получает параметры этой модели
optimizer = optim.SGD(model.parameters(), lr=lr)

# Средний квадрат ошибки
loss_fn = nn.MSELoss(reduction='mean')

# Создаём шаг для этой модели, функции потерь и оптимизатора
train_step_fn = make_train_step_fn(model, loss_fn, optimizer)

# Создаём шаг валидации
val_step_fn = make_val_step_fn(model, loss_fn)

# Создаём writer для TensorBoard
writer = SummaryWriter(mkdtemp(prefix='regression-', dir=tensorboard_dir))

# Берём батч для add_graph
x_sample, y_sample = next(iter(train_loader))
writer.add_graph(model, x_sample.to(device))

epoch_offset = 0

In [ ]:
%run -i "{lesson_dir}/model_configuration/v3.py"

### Обучение V5

In [ ]:
%%writefile "{lesson_dir}/model_training/v5.py"

# Количество эпох
n_epochs = 200

losses = []
val_losses = []

for epoch in range(n_epochs):
    # Внутренний цикл по батчам
    loss = mini_batch(device, train_loader, train_step_fn)
    losses.append(loss)
    
    # Валидация
    # При валидации граф не записываем
    with torch.no_grad():
        val_loss = mini_batch(device, val_loader, val_step_fn)
        val_losses.append(val_loss)
    
    # Записываем обе потери каждой эпохи под тегом loss
    writer.add_scalars(main_tag='loss',
                       tag_scalar_dict={'training': loss, 'validation': val_loss},
                       global_step=epoch_offset + epoch)

# Закрываем writer
writer.close()

In [ ]:
%run -i "{lesson_dir}/model_training/v5.py"

In [ ]:
# Проверяем параметры модели
print(model.state_dict())

<a id="section-8"></a>

## 8. Сохранение, возобновление и предсказания

Для предсказаний нужны структура модели и её состояние. Для продолжения обучения добавим состояние оптимизатора, число завершённых эпох и историю метрик. Загружайте только файлы из доверенного источника.

### Сохранение

In [ ]:
checkpoint = {'epoch': n_epochs,
              'model_state_dict': model.state_dict(),
              'optimizer_state_dict': optimizer.state_dict(),
              'loss': losses,
              'val_loss': val_losses}

torch.save(checkpoint, lesson_dir / 'model_checkpoint.pth')

### Возобновление обучения

In [ ]:
%run -i "{lesson_dir}/data_preparation/v2.py"
%run -i "{lesson_dir}/model_configuration/v3.py"

In [ ]:
print(model.state_dict())

In [ ]:
checkpoint = torch.load(lesson_dir / 'model_checkpoint.pth', map_location=device, weights_only=True)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

saved_epoch = checkpoint['epoch']
saved_losses = checkpoint['loss']
saved_val_losses = checkpoint['val_loss']

model.train() # Режим обучения при возобновлении
epoch_offset = saved_epoch

Восстановлены веса, состояние оптимизатора и номер эпохи. Это продолжение
обучения, но не обещание побитового совпадения с непрерывным запуском: для него
понадобились бы также состояния генераторов случайных чисел и порядок данных.
В новом журнале номера эпох продолжаются с `saved_epoch`.

In [ ]:
print(model.state_dict())

In [ ]:
%run -i "{lesson_dir}/model_training/v5.py"

In [ ]:
print(model.state_dict())

In [ ]:
fig = plot_resumed_losses(saved_epoch, saved_losses, saved_val_losses, n_epochs, losses, val_losses)

### Предсказания

Создаём модель той же структуры, восстанавливаем веса, включаем `eval()` и отключаем запись графа. Веса берём из сохранённой точки, не из последующего дообучения.

In [ ]:
device = select_device()
model = nn.Sequential(nn.Linear(1, 1)).to(device)

In [ ]:
checkpoint = torch.load(lesson_dir / 'model_checkpoint.pth', map_location=device, weights_only=True)

model.load_state_dict(checkpoint['model_state_dict'])

print(model.state_dict())

In [ ]:
new_inputs = torch.tensor([[.20], [.34], [.57]], dtype=torch.float32)

model.eval()
with torch.no_grad():
    predictions = model(new_inputs.to(device))
print(predictions.detach().cpu())

<a id="section-9"></a>

## 9. Собираем всё вместе

Подготовка данных, конфигурация модели и цикл обучения. Ниже те же три части без переходов между файлами. Новый класс-обёртка пока не нужен.

### Общая схема организации обучения

<img src="https://raw.githubusercontent.com/SergeyMalashenko/2026_ML3/main/notebooks/03_neuralnetwork/images/seminar_3_overall.png" alt="Общая схема организации обучения" width="1000">

[Открыть схему в полном размере](https://raw.githubusercontent.com/SergeyMalashenko/2026_ML3/main/notebooks/03_neuralnetwork/images/seminar_3_overall.png)

`zero_grad()` очищает накопленные градиенты; по умолчанию в текущем PyTorch это установка `.grad=None`, а не обязательно тензора нулей. Перед первым backward градиенты также должны быть очищены. Стрелки в правой части показывают связи компонентов, а не обязательную последовательность вызовов.

In [ ]:
# %load data_preparation/v2.py

torch.manual_seed(13)

# Преобразуем в тензоры до разбиения
x_tensor = torch.from_numpy(x).float()
y_tensor = torch.from_numpy(y).float()

# Dataset для всей выборки
dataset = TensorDataset(x_tensor, y_tensor)

# Разделяем индексы
ratio = .8
n_total = len(dataset)
n_train = int(n_total * ratio)
n_val = n_total - n_train

train_data, val_data = random_split(dataset, [n_train, n_val])

# Загрузчик для каждой части
train_loader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(dataset=val_data, batch_size=16)

In [ ]:
# %load model_configuration/v3.py

device = select_device()

# Скорость обучения
lr = 0.1

torch.manual_seed(42)
# Создаём модель и переносим её на выбранное устройство
model = nn.Sequential(nn.Linear(1, 1)).to(device)

# Оптимизатор получает параметры этой модели
optimizer = optim.SGD(model.parameters(), lr=lr)

# Средний квадрат ошибки
loss_fn = nn.MSELoss(reduction='mean')

# Создаём шаг для этой модели, функции потерь и оптимизатора
train_step_fn = make_train_step_fn(model, loss_fn, optimizer)

# Создаём шаг валидации
val_step_fn = make_val_step_fn(model, loss_fn)

# Создаём writer для TensorBoard
writer = SummaryWriter(mkdtemp(prefix='regression-', dir=tensorboard_dir))

# Берём батч для add_graph
x_sample, y_sample = next(iter(train_loader))
writer.add_graph(model, x_sample.to(device))

epoch_offset = 0

In [ ]:
# %load model_training/v5.py

# Количество эпох
n_epochs = 200

losses = []
val_losses = []

for epoch in range(n_epochs):
    # Внутренний цикл по батчам
    loss = mini_batch(device, train_loader, train_step_fn)
    losses.append(loss)
    
    # Валидация
    # При валидации граф не записываем
    with torch.no_grad():
        val_loss = mini_batch(device, val_loader, val_step_fn)
        val_losses.append(val_loss)
    
    # Записываем обе потери каждой эпохи под тегом loss
    writer.add_scalars(main_tag='loss',
                       tag_scalar_dict={'training': loss, 'validation': val_loss},
                       global_step=epoch_offset + epoch)

# Закрываем writer
writer.close()

In [ ]:
print(model.state_dict())

In [ ]:
evaluation_train_loader = DataLoader(train_data, batch_size=16, shuffle=False)
with torch.no_grad():
    final_train_mse = mini_batch(device, evaluation_train_loader, val_step_fn)
    final_val_mse = mini_batch(device, val_loader, val_step_fn)
print(f'MSE при финальных весах: train={final_train_mse:.6f}, validation={final_val_mse:.6f}')

<a id="section-10"></a>

## 10. К ДЗ1: данные и формы

Этот блок дополняет Chapter 2; условия выданной ДЗ1 не меняются.
`Dataset[i]` возвращает **один** пример, `collate_fn` получает **список примеров**,
а не заранее разделённые списки изображений и меток. Результат является батчем.

В ДЗ1 изображение до преобразования имеет форму `(28, 28)` и dtype `np.float32`.
Метка является индексом класса; в батче для NLL/CrossEntropy нужен `torch.int64`.
Ниже два маленьких искусственных изображения: проверяем упаковку, не качество модели.

**Вопросы:** какая форма появится после упаковки? После `Flatten`? Почему метки
не должны иметь форму `(B, 10)`, если выбран интерфейс с индексами классов?

In [ ]:
from torch.utils.data import default_collate

items = [(np.zeros((28, 28), dtype=np.float32), 2),
         (np.ones((28, 28), dtype=np.float32), 7)]
images, labels = default_collate(items)
print('Изображения:', images.shape, images.dtype)
print('Метки:', labels.shape, labels.dtype)
print('После Flatten:', nn.Flatten()(images).shape)
assert labels.dtype == torch.int64

В ДЗ1 встроенная FashionMNIST служит для загрузки исходных данных, а свой
Dataset и преобразования реализуются по условию. Нормировка пикселей и выбор
dtype являются разными действиями: преобразование типа само по себе не делит на 255.

Стандартный набор `train=False` используется в этой работе как **validation** во
всех сравнениях. Дополнительного третьего разбиения не требуется; независимую
финальную тестовую оценку здесь не проводим. `random_split` из основной части
показывает общий инструмент, но не добавляет новое требование к домашке.

<a id="section-11"></a>

## 11. К ДЗ1: собственный backward

Для обычного обучения достаточно операций PyTorch. В ДЗ1 собственный backward
нужен как упражнение. Возьмём пример сигмоиды из прошлогоднего семинара, не один
из заданных в ДЗ1 слоёв.

Если $z=\sigma(x)$, входящий `grad_output` содержит $\partial L/\partial z$.
`backward` возвращает $\partial L/\partial x$, а не одну лишь локальную производную.
Для поэлементной функции это произведение входящего градиента и локальной производной.
Для общего векторного преобразования это произведение на якобиан, не обязательно поэлементное.

**Вопрос:** если `grad_output` равен нулю, может ли этот путь дать ненулевой вклад?

In [ ]:
class MySigmoid(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        val = torch.sigmoid(x)
        ctx.save_for_backward(val)
        return val

    @staticmethod
    def backward(ctx, grad_output):
        (val,) = ctx.saved_tensors
        return grad_output * val * (1 - val)

sigmoid = MySigmoid.apply

In [ ]:
check_x = torch.tensor([-1., 0., 2.], dtype=torch.float64,
                       device='cpu', requires_grad=True)
assert torch.autograd.gradcheck(sigmoid, (check_x,))
incoming = torch.tensor([2., 0., -3.], dtype=torch.float64)
manual = torch.autograd.grad(sigmoid(check_x), check_x, incoming)[0]
reference = torch.autograd.grad(torch.sigmoid(check_x), check_x, incoming)[0]
torch.testing.assert_close(manual, reference)
print('Входящий градиент:', incoming)
print('Градиент по входу:', manual)

**Что проверять в ДЗ1:**

- Для каждого входа `forward` вернуть соответствующий градиент или `None`, если он не нужен. Формы должны совпадать с формами входов, включая bias после broadcasting.
- Для ReLU в нуле фиксируем значение 0; обычной производной там нет. Центральная разность даст $1/2$, поэтому `gradcheck` проводим вдали от излома.
- Для Dropout проверяем одну и ту же маску при численных возмущениях. Иначе сравниваются разные функции. Режим `eval()` проверяется отдельно.
- В `LogSoftmax` проверяем численную устойчивость на больших логитах. Ограничение на `grad_output` из условия сохраняется; полный якобиан хранить не требуется.
- `gradcheck` проверяет согласованность forward и backward, а не только наличие `.grad`. Правильность самого forward дополнительно сравниваем с эталонной операцией.

Обучение использует float32 и выбранное устройство. Численная проверка здесь
явно выполняется на CPU с float64, в том числе на компьютере с MPS.

<a id="section-12"></a>

## 12. К ДЗ1: классификация и эксперименты

Встроенная `nn.CrossEntropyLoss` принимает **логиты**. Пара `nn.LogSoftmax(dim=1)`
и `nn.NLLLoss` принимает тот же сигнал в два этапа. В ДЗ1 модуль с именем
`CrossEntropy` по условию принимает **логарифмы вероятностей**, то есть имеет
интерфейс NLLLoss. Не применяйте LogSoftmax второй раз внутри потерь.

Ниже используем готовые модули только для проверки интерфейсов. Это не реализация
слоёв ДЗ1 и не эксперимент на FashionMNIST. Веса и батч искусственные.

In [ ]:
logits = torch.tensor([[2., 0., -1.], [-1., 1., 2.]], device=device)
class_indices = torch.tensor([0, 2], dtype=torch.int64, device=device)
log_probabilities = nn.LogSoftmax(dim=1)(logits)
ce = nn.CrossEntropyLoss()(logits, class_indices)
nll = nn.NLLLoss()(log_probabilities, class_indices)
torch.testing.assert_close(ce, nll)
print('CrossEntropy по логитам:', ce.item())
print('NLL по log probabilities:', nll.item())

In [ ]:
torch.manual_seed(42)
classification_model = nn.Sequential(
    nn.Flatten(), nn.Linear(28 * 28, 32), nn.ReLU(),
    nn.Dropout(p=0.2), nn.Linear(32, 10),
).to(device)
classification_optimizer = optim.Adam(classification_model.parameters())
classification_loss = nn.CrossEntropyLoss()
classification_step = make_train_step_fn(
    classification_model, classification_loss, classification_optimizer)
print('Потеря одного учебного шага:',
      classification_step(images.to(device), labels.to(device)))
classification_model.eval()
with torch.no_grad():
    class_logits = classification_model(images.to(device))
print('Форма выхода:', class_logits.shape)
assert class_logits.shape == (2, 10)

**Вопросы перед экспериментами:**

- Почему несколько Linear без нелинейностей не дают многослойную нелинейную модель?
- Что должно быть одинаковым при сравнении своей реализации слоя со встроенной?
- Достаточно ли одного удачного seed, чтобы сделать вывод о глубине?

Сначала проверяем один батч, формы, конечность потерь и градиентов. Затем обучаем
на настоящих данных из ДЗ1, записывая train/validation loss и accuracy.
Сравниваем глубины по повторным запускам при оговорённых условиях, сохраняем
каждый запуск отдельно. Число слоёв, ширина, seed, learning rate и число эпох
должны быть записаны. Не выдаём результат на двух искусственных изображениях
за качество на FashionMNIST.

Для accuracy накапливаем число верных ответов и число объектов. Для среднего
loss учитываем размер каждого батча. Условия, бонусы и критерии берём из
[выданной ДЗ1](../../assignments/hw01.ipynb); этот раздел их не заменяет.

<a id="section-13"></a>

## 13. Задания

Каждое задание содержит собственные данные и проверку. Ответ раскрывайте после попытки. Основной материал выше выполняется без заполнения этих пропусков.

### T1. Согласованные примеры

Создайте TensorDataset `pairs` из `features` и `targets`. Сколько примеров он содержит?

<details>
<summary>Ответ</summary>

<pre><code class="language-python">pairs = TensorDataset(features, targets)</code></pre>

</details>

In [ ]:
features = torch.arange(6, dtype=torch.float32).reshape(3, 2)
targets = torch.tensor([2, 0, 1])

In [ ]:
raise NotImplementedError('T1: заполните пропуск')

In [ ]:
assert len(pairs) == 3
torch.testing.assert_close(pairs[1][0], features[1])
assert pairs[1][1].item() == 0
print('T1: проверка пройдена')

### T2. Последний батч

Создайте `loader` с batch_size=2, без перемешивания и без удаления последнего батча.

<details>
<summary>Ответ</summary>

<pre><code class="language-python">loader = DataLoader(batch_data, batch_size=2, shuffle=False, drop_last=False)</code></pre>

</details>

In [ ]:
batch_data = TensorDataset(torch.arange(5))

In [ ]:
raise NotImplementedError('T2: заполните пропуск')

In [ ]:
batches = list(loader)
assert [len(b[0]) for b in batches] == [2, 2, 1]
assert torch.cat([b[0] for b in batches]).tolist() == [0, 1, 2, 3, 4]
print('T2: проверка пройдена')

### T3. Среднее по объектам

Батчевые средние равны 1 и 9; размеры 16 и 4. Вычислите `epoch_mean`.

<details>
<summary>Ответ</summary>

<pre><code class="language-python">epoch_mean = np.average(batch_means, weights=batch_sizes)</code></pre>

</details>

In [ ]:
batch_means = np.array([1., 9.])
batch_sizes = np.array([16, 4])

In [ ]:
raise NotImplementedError('T3: заполните пропуск')

In [ ]:
assert np.isclose(epoch_mean, 2.6)
print('T3: проверка пройдена')

### T4. Два переключателя

Переведите `probe` в eval и вычислите `probe_output` без записи графа.

<details>
<summary>Ответ</summary>

<pre><code class="language-python">probe.eval()
with torch.no_grad():
    probe_output = probe(probe_input)</code></pre>

</details>

In [ ]:
probe = nn.Sequential(nn.Linear(2, 2), nn.Dropout(p=0.5))
probe_input = torch.ones(3, 2)
probe.train()

In [ ]:
raise NotImplementedError('T4: заполните пропуск')

In [ ]:
assert not probe.training and not probe[1].training
assert not probe_output.requires_grad
torch.testing.assert_close(probe_output, probe[0](probe_input).detach())
print('T4: проверка пройдена')

### T5. Восстановить веса

Загрузите `saved_state` в `restored_model` без нового обучения.

<details>
<summary>Ответ</summary>

<pre><code class="language-python">restored_model.load_state_dict(saved_state)</code></pre>

</details>

In [ ]:
torch.manual_seed(7)
original_model = nn.Linear(2, 1)
saved_state = {name: value.detach().clone() for name, value in original_model.state_dict().items()}
restored_model = nn.Linear(2, 1)

In [ ]:
raise NotImplementedError('T5: заполните пропуск')

In [ ]:
for name, parameter in restored_model.state_dict().items():
    torch.testing.assert_close(parameter, saved_state[name])
print('T5: проверка пройдена')

### T6. Входящий градиент

Для z = sigmoid(x) в x=0 входящий градиент равен 3. Вычислите `grad_input`.

<details>
<summary>Ответ</summary>

<pre><code class="language-python">grad_input = grad_output * sigmoid_value * (1 - sigmoid_value)</code></pre>

</details>

In [ ]:
sigmoid_value = torch.tensor(0.5)
grad_output = torch.tensor(3.)

In [ ]:
raise NotImplementedError('T6: заполните пропуск')

In [ ]:
torch.testing.assert_close(grad_input, torch.tensor(0.75))
print('T6: проверка пройдена')

## Материалы

Daniel Voigt Godoy, [Chapter 2](https://github.com/dvgodoy/PyTorchStepByStep/blob/master/Chapter02.ipynb).
Основной код и композиция графиков адаптированы из PyTorchStepByStep,
Copyright (c) 2020 Daniel Voigt Godoy, [лицензия MIT](../../LICENSE-GODOY).
Порядок основной главы сохранён. MPS, временные файлы, взвешивание батчей,
безопасная загрузка собственного checkpoint, дополнения к ДЗ1 и T1–T6 добавлены для курса.

[Dataset и DataLoader](https://docs.pytorch.org/docs/stable/data.html),
[собственный autograd.Function](https://docs.pytorch.org/docs/stable/notes/extending.html),
[TensorBoard](https://docs.pytorch.org/docs/stable/tensorboard.html),
[сохранение и загрузка](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).